##**Projeto:** Merca Data Platform
##**Squad:** 2 | Camada Silver
### Objetivo
Processar os dados do catálogo de produtos da Bronze, aplicar três regras de qualidade e gravar os registros válidos na camada Silver de forma incremental.
### Origem e Destino
| Item | Valor |
| **Origem** | `squad2/bronze/ecommerce_produtos` (Delta Lake) |
| **Destino** | `squad2/silver/ecommerce_produtos` (Delta Lake) |
| **Controle** | `silver/control/ecommerce_produtos.json` |
| **Modo de escrita** | `append` incremental por arquivo de origem |
### Regras de Qualidade Aplicadas
| # | Campo(s) | Regra | Ação quando falha |
| 1 | `sku` | Não nulo; comprimento entre **6 e 59 caracteres** | Linha descartada |
| 2 | `preco_lista` | Maior que `0` e menor que `5.000` | Linha descartada |
| 3 | `is_ativo` | Não nulo; convertido para tipo `bool` | Linha descartada |
As três condições são aplicadas **simultaneamente** — um produto precisa passar nas três para ser promovido à Silver.
### Coluna de Auditoria Adicionada
| Coluna | Descrição |
| `silver_processed_at` | Timestamp de processamento na Silver |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, `get_storage_options`, `get_squad2_client` |

In [0]:
%run ../utils/feat_squad2_99_helpers

- Configuração de Caminhos e Controle Incremental

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_produtos"

# Caminhos ABFSS oficiais para o Delta Lake
path_bronze = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/{TABELA}"
path_silver = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"

# Caminho do arquivo de controle dentro do container
path_control = f"silver/control/{TABELA}.json"

print(f" Lendo de (Bronze): {path_bronze}")
print(f" Gravando em (Silver): {path_silver}")


- Leitura da Bronze, Aplicação das Regras e Gravação na Silver
**Controle incremental:** apenas linhas cujo `bronze_source_file` ainda não foi processado
**Regras aplicadas em sequência:**
- **Regra 1 — SKU válido:** deve estar preenchido e ter entre 6 e 59 caracteres.
- **Regra 2 — Preço realista:** valor de lista entre R$ 0,01 e R$ 4.999,99. Elimina produtos com preço zerado, negativo ou absurdamente alto (erro de cadastro).
- **Regra 3 — Status booleano presente:** campo `is_ativo` não pode ser nulo; é convertido para `bool` puro para garantir consistência de tipo na Silver.

In [0]:
try:
    # 1. Abre a tabela Bronze e converte para Pandas
    dt_bronze = DeltaTable(path_bronze, storage_options=get_storage_options())
    df_pandas = dt_bronze.to_pandas()
    
    # 2. CONTROLE INCREMENTAL: Instancia o cliente do arquivo de controle
    squad2_client = get_squad2_client()
    file_client = squad2_client.get_file_client(path_control)
    
    processados = set()
    
    # Se o arquivo JSON já existir na pasta silver/control, baixa e lê o conteúdo
    if file_client.exists():
        conteudo = file_client.download_file().readall().decode('utf-8')
        processados = set(json.loads(conteudo))
    
    # Filtra apenas os dados de arquivos que a Silver ainda não processou
    df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
    
    if df_novos_dados.empty:
        print(" Camada Silver em dia! Nenhum dado novo para processar.")
    else:
        print(f" Processando {len(df_novos_dados)} novas linhas da Bronze...")
        
        # 3. APLICAÇÃO DAS REGRAS TÉCNICAS DA PLANILHA DE PRODUTOS
        # Regra Técnica 1: SKU válido (não nulo e com comprimento maior que 5 e menor que 60)
        cond_sku = (df_novos_dados['sku'].notna()) & \
                   (df_novos_dados['sku'].astype(str).str.len() > 5) & \
                   (df_novos_dados['sku'].astype(str).str.len() < 60)
        
        # Regra Técnica 2: Limites realistas para produtos secos (preco_lista > 0 e < 5000)
        cond_preco = (df_novos_dados['preco_lista'] > 0) & (df_novos_dados['preco_lista'] < 5000)
        
        # Regra Técnica 3: Status disponível sem nulos
        cond_ativo = df_novos_dados['is_ativo'].notna()
        
        # Filtragem com base nas três condições técnicas simultâneas
        df_filtrado = df_novos_dados[cond_sku & cond_preco & cond_ativo].copy()
        
        # Força a conversão estrita para garantir o tipo booleano limpo na Silver
        df_filtrado['is_ativo'] = df_filtrado['is_ativo'].astype(bool)
        
        # 4. COLUNA DE AUDITORIA
        df_filtrado['silver_processed_at'] = datetime.now()
        
        # Remove os fuso-horários para o formato Delta
        for col in df_filtrado.columns:
            if pd.api.types.is_datetime64_any_dtype(df_filtrado[col]):
                df_filtrado[col] = df_filtrado[col].dt.tz_localize(None)
                
        # 5. GRAVAÇÃO NA SILVER VIA DELTA
        write_deltalake(
            table_or_uri    = path_silver,
            data            = df_filtrado,
            mode            = "append",
            storage_options = get_storage_options()
        )
        
        # 6. ATUALIZA O CONTROL JSON USANDO O FILE CLIENT
        arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
        todos_processados = list(processados.union(arquivos_atuais))
        
        # upload_data com overwrite=True cria o arquivo ou atualiza se já existir
        file_client.upload_data(json.dumps(todos_processados), overwrite=True)
        
        print(f" SUCESSO! {len(df_filtrado)} linhas filtradas e salvas com a control atualizada no Azure!")

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise